# Sales Data Validation and Cleaning

## Objective:
To validate and clean the sales dataset by addressing data quality issues such as missing values, duplicates, inconsistent entries, and incorrect data types, ensuring that the dataset is accurate, consistent, and ready for analysis and visualization.

In [41]:
#Importing Libraries
import pandas as pd
import numpy as np

In [42]:
#Importing Dataset
df = pd.read_csv("../dataset/retail_store_sales.csv")

In [43]:
#Displaying first 5 rows
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [44]:
#Checking the dimensions of the dataset
df.shape

(12575, 11)

## Data Cleaning

In [45]:
#Getting information about the dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  str    
 1   Customer ID       12575 non-null  str    
 2   Category          12575 non-null  str    
 3   Item              11362 non-null  str    
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  str    
 8   Location          12575 non-null  str    
 9   Transaction Date  12575 non-null  str    
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(1), str(7)
memory usage: 1.9+ MB


## Handling Missing Values

In [46]:
#Checking for missing values
df.isnull().sum()

Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

In [47]:
#Checking counts in discount column
df['Discount Applied'].value_counts(dropna=False)

Discount Applied
True     4219
NaN      4199
False    4157
Name: count, dtype: int64

The Discount Applied column contained 4,199 missing values. Since there was insufficient information to determine whether a discount had been applied, the missing values were replaced with "Unknown" to preserve the records without introducing assumptions.

In [48]:
#Filling missing values of Discount Applied column
df['Discount Applied'] = df['Discount Applied'].fillna('Unknown')

The Item column contained 1,213 missing values. Since the missing product names could not be reliably inferred from the available information, the missing values were replaced with "Unknown" to preserve the associated transaction records.

In [49]:
#Filling missing values of Items column
df['Item'] = df['Item'].fillna('Unknown')

### Recovering Missing Values in Related Numerical Columns

The columns `Price Per Unit`, `Quantity`, and `Total Spent` are mathematically related. Missing values in these columns were recovered using the information available in the other two columns, allowing the affected records to be retained without relying on statistical imputation.

In [50]:
#Filling missing values in Price Per Unit column
mask = (
    df['Price Per Unit'].isnull() &
    df['Quantity'].notnull() &
    df['Total Spent'].notnull()
)

df.loc[mask, 'Price Per Unit'] = (
    df.loc[mask, 'Total Spent'] /
    df.loc[mask, 'Quantity']
)

In [51]:
#Filling missing values in Quantity column
mask = (
    df['Quantity'].isnull() &
    df['Price Per Unit'].notnull() &
    df['Total Spent'].notnull()
)

df.loc[mask, 'Quantity'] = (
    df.loc[mask, 'Total Spent'] /
    df.loc[mask, 'Price Per Unit']
)

In [52]:
#Filling missing values in Total Spent Column
mask = (
    df['Price Per Unit'].notnull() &
    df['Quantity'].notnull() &
    df['Total Spent'].isnull()
)

df.loc[mask, 'Total Spent'] = (
    df.loc[mask, 'Quantity'] *
    df.loc[mask, 'Price Per Unit']
)

In [53]:
#Validating remaining missing values in Price Per Unit, Quantity, and Total Spent columns
missing = df[['Price Per Unit','Quantity','Total Spent']].isnull().sum()
print(missing)

Price Per Unit      0
Quantity          604
Total Spent       604
dtype: int64


The `Quantity` and `Total Spent` columns contained 604 missing values occurring in the same records. Since both variables were missing simultaneously, there was insufficient information to accurately recover the missing values. Therefore, these records were removed to maintain the quality and reliability of the dataset.

In [54]:
df = df.dropna(subset=['Quantity','Total Spent'])

## Handling Duplicate Records

In [55]:
df.duplicated().sum()

np.int64(0)

The dataset was examined for duplicate records to ensure the uniqueness and integrity of the transaction data. No duplicate entries were identified, indicating that each record represents a distinct transaction.

## Data Type Correction
The data types of all columns were reviewed to ensure consistency and suitability for analysis. During this process, the Transaction Date column was identified as having an incorrect data type (object). The column was converted to a datetime format to enable accurate time-based analysis and improve overall data quality.

In [56]:
#Correcting data type of Transaction Date column
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])

## Final Data Validation
Following the preprocessing steps, the dataset was validated to confirm the successful handling of missing values, the absence of duplicate records, and the consistency of data types. The cleaned dataset was then exported for further analysis and visualization.

In [57]:
#Checking Dimensions
df.shape

(11971, 11)

In [58]:
#Checking Missing Values
df.isnull().sum()

Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64

In [59]:
#Checking Duplicates
df.duplicated().sum()

np.int64(0)

In [60]:
#Dataset Information
df.info()

<class 'pandas.DataFrame'>
Index: 11971 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    11971 non-null  str           
 1   Customer ID       11971 non-null  str           
 2   Category          11971 non-null  str           
 3   Item              11971 non-null  str           
 4   Price Per Unit    11971 non-null  float64       
 5   Quantity          11971 non-null  float64       
 6   Total Spent       11971 non-null  float64       
 7   Payment Method    11971 non-null  str           
 8   Location          11971 non-null  str           
 9   Transaction Date  11971 non-null  datetime64[us]
 10  Discount Applied  11971 non-null  object        
dtypes: datetime64[us](1), float64(3), object(1), str(6)
memory usage: 1.8+ MB


## Exporting Dataset

In [61]:
#Exporting the cleaned Dataset
df.to_csv('cleaned_data.csv', index=False)

## Conclusion
The dataset was successfully cleaned and validated through a series of preprocessing steps, including handling missing values, removing incomplete records, checking for duplicate entries, and correcting data types. Final validation and summary statistics confirmed the consistency and quality of the data. The cleaned dataset was then exported and prepared for further analysis and visualization.

- Original Dataset Rows: 12,575
- Cleaned Dataset Rows: 11,971